# Pipeline Control Utilities (utils.ipynb)
Provides reusable functions for metadata lookup, watermark tracking, and status updating in Medallion Lakehouse pipelines.

In [ ]:
# 1. Initialize SparkSession
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
%run ./logger

In [ ]:
# 2. Initialize Centralized Logger for Utilities
logger = get_task_logger(notebook_name_override="pipeline_utils")

In [ ]:
# 3. Extract Metadata by Table ID or Table Name
def extract_table_metadata(table_id_or_name):
    """
    Fetches pipeline metadata from spark_training.metadata_schema.metadata_table.
    Accepts either an integer table_id (e.g. 101) or a string table_name (e.g. 'countries').
    Returns a dictionary of metadata attributes for clean, easy access.
    """
    if isinstance(table_id_or_name, int) or (isinstance(table_id_or_name, str) and table_id_or_name.isdigit()):
        filter_clause = f"table_id = {int(table_id_or_name)}"
    else:
        filter_clause = f"lower(source_table_name) = lower('{table_id_or_name}')"

    query = f"""
        SELECT *
        FROM spark_training.metadata_schema.metadata_table
        WHERE {filter_clause}
    """
    df = spark.sql(query)
    rows = df.collect()
    if not rows:
        raise ValueError(f"No metadata found matching: '{table_id_or_name}' in spark_training.metadata_schema.metadata_table")
    return rows[0].asDict()

In [ ]:
# 4. Get All Active Tables for a Target Layer
def get_active_tables(target_layer: str = "bronze"):
    """
    Retrieves all active tables configured for a specific target layer (e.g. 'bronze', 'silver').
    Returns a list of dictionaries, ordered by table_id.
    """
    query = f"""
        SELECT *
        FROM spark_training.metadata_schema.metadata_table
        WHERE is_active = 'true' AND lower(target_name) = lower('{target_layer}')
        ORDER BY table_id
    """
    df = spark.sql(query)
    return [row.asDict() for row in df.collect()]

In [ ]:
# 5. Atomically Update Pipeline Metadata (Watermark and/or Status)
def update_pipeline_metadata(table_id: int, status: str = None, watermark = None):
    """
    Atomically updates last_run_status and/or last_load_date watermark in a single Delta Lake transaction.
    Significantly reduces Delta commit overhead by avoiding multiple consecutive writes.
    """
    try:
        set_clauses = ["updated_on = CURRENT_DATE()", "updated_by = 'sham'"]
        if status is not None:
            set_clauses.append(f"last_run_status = '{status}'")
        if watermark is not None:
            set_clauses.append(f"last_load_date = '{watermark}'")
        
        set_str = ", ".join(set_clauses)
        query = f"""
            UPDATE spark_training.metadata_schema.metadata_table
            SET {set_str}
            WHERE table_id = {table_id}
        """
        spark.sql(query)
        logger.info(f"Updated metadata for table_id {table_id} (status={status}, watermark={watermark})")
        return f"Successfully updated metadata for table_id {table_id}"
    except Exception as e:
        logger.error(f"Failed to update metadata for table_id {table_id}: {e}")
        raise e

def update_last_load_date(table_id: int, last_load_date):
    """Updates the last_load_date watermark."""
    return update_pipeline_metadata(table_id, watermark=last_load_date)

def update_last_load_status(table_id: int, status: str):
    """Updates the last_run_status ('SUCCESS', 'FAILED', 'RUNNING')."""
    return update_pipeline_metadata(table_id, status=status)


In [ ]:
# 7. Interactive Test & Verification (uncomment to test directly in notebook)
# meta = extract_table_metadata(101)
# print(f"Test Metadata: ID {meta['table_id']} -> {meta['source_table_name']} (Pushdown: {meta['query'][:40]}...)")
# print(f"Active Bronze Tables Count: {len(get_active_tables('bronze'))}")